<a href="https://colab.research.google.com/github/memo124/Tareas---KODIGO/blob/main/welcome_to_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Paso 0: Agregar importaciones de librerias

In [ ]:
from abc import ABC, abstractmethod
from pathlib import Path
import logging

import pandas as pd
import matplotlib.pyplot as plt

## Paso 1: Clase Abstracta

In [ ]:
class DataComponent(ABC):
  @abstractmethod
  def execute(self):
    pass

## Paso 2: Mixins (Logger y Validator)

In [ ]:
logging.basicConfig(
    level = logging.INFO,
    format = "[%(asctime)s] [%(levelname)s - %(name)s] %(message)s"
)

class LoggerMixin:
    _logger: logging.Logger | None = None
    
    @property
    def logger(self) -> logging.Logger:
        return self._logger or logging.getLogger(type(self).__name__)
    
    def log_info(self, message):
        self.logger.info(message)

    def log_warning(self, message):
        self.logger.warning(message)

    def log_error(self, message):
        self.logger.error(message)
        
class ValidatorMixin:
    
    def validate_file(self, file_path):
        # Verifica que la ruta corresponda a un archivo existente
        if not Path(file_path).is_file():
            raise FileNotFoundError(
                f"No existe el archivo: {file_path}"
            )
            
    def validate_columns(self, dataframe, required_columns):
        # Encuentra las columnas requeridas que no existen en el DataFrame
        missing_columns = [
            column
            for column in required_columns
            if column not in dataframe.columns
        ]

        # Detiene el proceso si falta al menos una columna
        if missing_columns:
            raise ValueError(
                f"Faltan las columnas: {missing_columns}"
            )
            
    def validate_nulls(self, dataframe, columns=None):
        columns = columns or dataframe.columns
        
        null_counts = dataframe[columns].isna().sum()
        columns_with_nulls = null_counts[null_counts > 0]

        # Detiene el proceso si se encuentran valores nulos
        if not columns_with_nulls.empty:
            raise ValueError(
                f"Columnas con valores nulos:\n{columns_with_nulls}"
            )

## Paso 3: Clase DataLoader

In [ ]:
class DataLoader(DataComponent, ValidatorMixin, LoggerMixin):
    def __init__(self, file_path, required_columns=None):
        self.__file_path = file_path
        self.__required_columns = required_columns or []
        
    def execute(self):
        try:
            self.log_info(f"Cargando archivo: {self.__file_path}")
            # Valida que el archivo exista antes de leerlo.
            self.validate_file(self.__file_path)
        
            # Carga el archivo CSV en un DataFrame.
            dataframe = pd.read_csv(self.__file_path)

            # Valida las columnas requeridas despues de cargar los datos.
            if self.__required_columns:
                self.validate_columns(dataframe, self.__required_columns)

            self.log_info(f"Archivo cargado correctamente: {len(dataframe)} filas")
            return dataframe
        except Exception as error:
            self.log_error(f"Error al cargar los datos: {error}")
            raise

## Paso 4: Clase DataCleaner

In [ ]:
class DataCleaner(DataComponent, LoggerMixin):
    # Componente de limpieza: infiere tipos e imputa valores faltantes.
    def __init__(self, dataframe, categorical_fill=None):
        # Guarda los datos recibidos y la configuracion opcional de limpieza.
        self.__dataframe = dataframe
        self.__categorical_fill = categorical_fill

    @property
    def summary(self):
        # Estadisticas descriptivas de solo lectura del dataset.
        return self.__dataframe.describe(include="all")

    @property
    def null_report(self):
        # Conteo de nulos por columna de solo lectura.
        return self.__dataframe.isna().sum()

    def execute(self):
        # Ejecuta el flujo completo de limpieza y retorna el DataFrame limpio.
        try:
            self.log_info("Iniciando limpieza de datos")
            self.__dataframe = self.__dataframe.copy()
            self.__convert_types()
            self.__impute_nulls()
            self.log_info("Limpieza de datos completada")
            return self.__dataframe
        except Exception as error:
            self.log_error(f"Error durante la limpieza de datos: {error}")
            raise

    def __convert_types(self):
        # Aplica la conversion automatica de tipos de Pandas.
        self.log_info("Convirtiendo tipos de datos")
        self.__dataframe = self.__dataframe.convert_dtypes()

    def __impute_nulls(self):
        # Rellena valores faltantes: mediana, valor configurado o moda.
        self.log_info("Imputando valores nulos")
        for column in self.__dataframe.columns:
            null_count = self.__dataframe[column].isna().sum()

            # Omite columnas sin valores faltantes.
            if null_count == 0:
                continue
            is_numeric = pd.api.types.is_numeric_dtype(self.__dataframe[column])
            mode_values = self.__dataframe[column].mode()

            # Las columnas numericas usan la mediana; las de texto usan un valor configurado o la moda.
            if is_numeric:
                fill_value = self.__dataframe[column].median()
            elif self.__categorical_fill is not None:
                fill_value = self.__categorical_fill
            else:
                fill_value = mode_values[0] if not mode_values.empty else "unknown"

            self.__dataframe[column] = self.__dataframe[column].fillna(fill_value)
            self.log_warning(f"Columna '{column}': {null_count} valores nulos imputados")

## Paso 5: DataAnalyst

In [ ]:
class DataAnalyst(DataComponent, LoggerMixin):
    # Tipos de reporte permitidos: console, graphic o both.
    def __init__(self, dataframe, report_type="both", graphic_columns=None, limit=30):
        self.__dataframe = dataframe
        self.__report_type = report_type
        self.__graphic_columns = graphic_columns
        self.__limit = limit

    def execute(self):
        self.log_info("Analizando los datos")
        
        try:
            # Valida que el tipo de reporte sea permitido.
            if self.__report_type not in ["console", "graphic", "both"]:
                raise ValueError("Tipo de reporte no valido")

            # Ejecuta el reporte de consola si fue solicitado.
            if self.__report_type in ["console", "both"]:
                self.__generate_console_report()
                
            # Ejecuta el reporte grafico si fue solicitado.
            if self.__report_type in ["graphic", "both"]:
                self.__generate_graphic_report()
                    
        except Exception as error:
            self.log_error(error)
            raise

    def __generate_console_report(self):
        # Muestra una vista de las primeras filas del dataset.
        columns = self.__graphic_columns or self.__dataframe.columns
        print(self.__dataframe[columns].head(self.__limit))

    def __generate_graphic_report(self):
        # Usa columnas numericas por defecto cuando no se seleccionan columnas.
        if self.__graphic_columns is None:
            self.__dataframe.hist(figsize=(10, 6))
            plt.show()
            return

        # Crea un grafico por cada columna seleccionada.
        for column in self.__graphic_columns:
            self.__dataframe[column].value_counts().head(self.__limit).plot(kind="bar")
            plt.title(column)
            plt.show()

## Paso 6: Main del Notebook

In [ ]:
file_path = "/content/exercises.csv" #@param {type:"string"}
categorical_fill = "unknown" #@param {type:"string"}
report_type = "both" #@param ["console", "graphic", "both"]
report_limit = 30 #@param {type:"integer"}
required_columns = ["category", "difficulty", "equipment"]

def create_loader(data):
    return DataLoader(file_path, required_columns)

def create_cleaner(data):
    return DataCleaner(data, categorical_fill)

def create_analyst(data):
    return DataAnalyst(data, report_type, required_columns, report_limit)

In [ ]:
try:
    # Pasos del pipeline ejecutados en orden.
    pipeline = [create_loader, create_cleaner, create_analyst]
    # Entrada inicial para el primer componente.
    data = None

    for create_component in pipeline:
        component = create_component(data)
        # Polimorfismo: cada componente se ejecuta con execute().
        result = component.execute()

        if result is not None:
            data = result
except Exception as error:
    print(f"Pipeline detenido: {error}")